# ASR Baseline (Whisper) – Zeroth Korean

이 노트북은 Zeroth Korean 테스트 세트를 대상으로 **Whisper ASR 베이스라인 성능(WER / CER)**을 빠르게 확인하기 위한 실험용 노트북입니다.

- 입력 오디오: `~/data_zeroth/audio/**/**/*.wav` (하위 디렉터리를 포함한 모든 WAV 파일)
- 정답 스크립트: `~/data_zeroth/refs.csv` (파일명 기준으로 오디오와 매칭)


In [1]:
!pip install -U "transformers>=4.41" accelerate jiwer soundfile librosa pandas

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import os, glob, time
import pandas as pd
import torch
from transformers import pipeline, AutoProcessor, AutoModelForSpeechSeq2Seq
from jiwer import wer, cer
from pathlib import Path

In [3]:
HOME = Path.home()

DATA_AUDIO_DIR = HOME / "data_zeroth" / "test_data_01"
REF_CSV = HOME / "data_zeroth" / "refs.csv"

MODEL_ID = "openai/whisper-base"

LANGUAGE = "korean"
TASK = "transcribe"

MAX_FILES = 50  # 빠른 확인용

print("MODEL_ID:", MODEL_ID)
print("DATA_AUDIO_DIR:", DATA_AUDIO_DIR)
print("REF_CSV:", REF_CSV)

MODEL_ID: openai/whisper-base
DATA_AUDIO_DIR: /home/data/bootcamp/user2/data_zeroth/test_data_01
REF_CSV: /home/data/bootcamp/user2/data_zeroth/refs.csv


In [4]:
import os, torch

if torch.cuda.is_available():
    # 현재 프로세스에 노출된 GPU는 항상 0..N-1로 재매핑됨
    device = 0
    visible = os.environ.get("CUDA_VISIBLE_DEVICES", "(not set)")
else:
    device = -1
    visible = "(cpu)"

print("CUDA_VISIBLE_DEVICES:", visible, "| pipeline device:", device, "| cuda_count:", torch.cuda.device_count())

dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print("CUDA:", torch.cuda.is_available(), "| device:", device)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForSpeechSeq2Seq.from_pretrained(MODEL_ID, dtype=dtype)
if torch.cuda.is_available():
    model = model.to("cuda")

asr = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=device,
)

print("ASR pipeline ready")

CUDA_VISIBLE_DEVICES: 1 | pipeline device: 0 | cuda_count: 1
CUDA: True | device: 0


Device set to use cuda:0


ASR pipeline ready


In [5]:
audio_files = sorted(glob.glob(os.path.join(DATA_AUDIO_DIR, "**", "*.wav"), recursive=True))
if MAX_FILES > 0:
    audio_files = audio_files[:MAX_FILES]

print("# audio files:", len(audio_files))
audio_files[:5]

# audio files: 50


['/home/data/bootcamp/user2/data_zeroth/test_data_01/003/104_003_0019.wav',
 '/home/data/bootcamp/user2/data_zeroth/test_data_01/003/104_003_0193.wav',
 '/home/data/bootcamp/user2/data_zeroth/test_data_01/003/104_003_0253.wav',
 '/home/data/bootcamp/user2/data_zeroth/test_data_01/003/104_003_0280.wav',
 '/home/data/bootcamp/user2/data_zeroth/test_data_01/003/104_003_0294.wav']

In [6]:
def transcribe_chunked(files, batch_size=8, step=10):
    all_rows = []
    t0 = time.time()
    for start in range(0, len(files), step):
        chunk = files[start:start+step]
        outs = asr(chunk, generate_kwargs={"language": LANGUAGE, "task": TASK}, batch_size=batch_size)
        for f, out in zip(chunk, outs):
            all_rows.append({"file": str(f), "fileid": os.path.basename(f), "hyp": out["text"].strip()})
        print(f"[{min(start+step, len(files))}/{len(files)}] done")
    df = pd.DataFrame(all_rows)
    print("Elapsed(s):", round(time.time() - t0, 2))
    return df

print("음성인식을 시작합니다")
df_hyp = transcribe_chunked(audio_files, batch_size=8)
df_hyp.head()
print("음성인식을 완료했습니다")

`return_token_timestamps` is deprecated for WhisperFeatureExtractor and will be removed in Transformers v5. Use `return_attention_mask` instead, as the number of frames can be inferred from it.


음성인식을 시작합니다
[10/50] done
[20/50] done
[30/50] done
[40/50] done
[50/50] done
Elapsed(s): 11.54
음성인식을 완료했습니다


In [7]:
import os

df_ref = pd.read_csv(REF_CSV)

print("df_ref columns:", df_ref.columns.tolist())
print("df_hyp columns:", df_hyp.columns.tolist())

# df_ref: file 컬럼 정규화
if "file" not in df_ref.columns:
    raise ValueError("refs.csv에서 file 컬럼을 찾지 못했습니다. columns=" + str(df_ref.columns.tolist()))
df_ref["file"] = df_ref["file"].astype(str).apply(os.path.basename)

# df_ref: transcript -> ref
if "ref" not in df_ref.columns:
    if "transcript" in df_ref.columns:
        df_ref = df_ref.rename(columns={"transcript": "ref"})
    else:
        raise ValueError("refs.csv에서 정답 텍스트 컬럼을 찾지 못했습니다. columns=" + str(df_ref.columns.tolist()))
df_ref["ref"] = df_ref["ref"].astype(str)

# df_hyp: fileid가 없으면 생성
if "fileid" not in df_hyp.columns:
    df_hyp = df_hyp.copy()
    df_hyp["fileid"] = df_hyp["file"].astype(str).apply(os.path.basename)

# merge: df_ref.file (basename)  <->  df_hyp.fileid (basename)
df = df_ref.merge(df_hyp, left_on="file", right_on="fileid", how="inner", suffixes=("_ref", "_hyp"))
df["wav_path"] = df["file_hyp"]   # df_hyp의 file(전체경로)

print("merged rows:", len(df), "/", len(df_ref), "(refs)")
if df.empty:
    print("예시 refs file:", df_ref["file"].head().tolist())
    print("예시 hyp fileid:", df_hyp["fileid"].head().tolist())
    raise ValueError("refs.csv의 file 값과 df_hyp의 fileid 값이 매칭되지 않습니다. 파일명 규칙을 맞춰야 합니다.")

print("WER:", round(wer(df["ref"].tolist(), df["hyp"].tolist()), 4))
print("CER:", round(cer(df["ref"].tolist(), df["hyp"].tolist()), 4))

df.head()


df_ref columns: ['file', 'transcript']
df_hyp columns: ['file', 'fileid', 'hyp']
merged rows: 50 / 456 (refs)
WER: 0.425
CER: 0.1478


,file_ref,ref,file_hyp,fileid,hyp,wav_path
0,104_003_0019.wav,지난해 삼 월 김 전 장관의 동료인 장동련 홍익대 교수가 민간 자문단장으로 위촉되면...,/home/data/bootcamp/user2/data_zeroth/test_dat...,104_003_0019.wav,지난해 3월 김 전 장관의 동료인 장동량 홍익대 교수가 민간 자문단장으로 위쪽되면서...,/home/data/bootcamp/user2/data_zeroth/test_dat...
1,104_003_0193.wav,그 바람에 나의 몸도 겹쳐 쓰러지며 한창 피어난 노란 동백꽃 속으로 폭 파묻혀 버렸다,/home/data/bootcamp/user2/data_zeroth/test_dat...,104_003_0193.wav,그 바람에 나의 몸도 겹쳐스러지면 한참 피어난 노란 동백꽃속으로 폭파무처버렸다.,/home/data/bootcamp/user2/data_zeroth/test_dat...
2,104_003_0253.wav,현재 백화점과 영화관 등은 오픈해 영업하고 있고 테마파크 및 아파트 등의 공사는 이...,/home/data/bootcamp/user2/data_zeroth/test_dat...,104_003_0253.wav,현재 백화점과 영화관 등은 오픈의 영화파고 있고 테마파크미 아파트 등의 공사는 20...,/home/data/bootcamp/user2/data_zeroth/test_dat...
3,104_003_0280.wav,또한 박 전 대통령 측은 앞으로 검찰 수사와 관련해 수사 내용에 대한 입장뿐 아니라...,/home/data/bootcamp/user2/data_zeroth/test_dat...,104_003_0280.wav,또한 박 전 대통령 측은 앞으로 검찰 수사와 관련해 수사 내용에 대한 입장 뿐 아니...,/home/data/bootcamp/user2/data_zeroth/test_dat...
4,104_003_0294.wav,한 잔의 커피가 소비자에게 전해지기 위해선 여러 차례의 공정을 거쳐야 한다,/home/data/bootcamp/user2/data_zeroth/test_dat...,104_003_0294.wav,한잔의 커피가 소비자에게 전해지기 위해서는 여러 차례의 공정을 거쳐야 한다.,/home/data/bootcamp/user2/data_zeroth/test_dat...


In [8]:
import os
from IPython.display import Audio, display

for _, r in df.iterrows():
    print("FILE:", r["file_ref"])
    print("REF:", r["ref"])
    print("HYP:", r["hyp"])
    print("PATH:", r["wav_path"])
    display(Audio(filename=r["wav_path"]))
    print("-" * 60)


FILE: 104_003_0019.wav
REF: 지난해 삼 월 김 전 장관의 동료인 장동련 홍익대 교수가 민간 자문단장으로 위촉되면서 본격적인 공모와 개발 작업에 들어갔다
HYP: 지난해 3월 김 전 장관의 동료인 장동량 홍익대 교수가 민간 자문단장으로 위쪽되면서 본격적인 공모와 개발 작업에 들어갔다.
PATH: /home/data/bootcamp/user2/data_zeroth/test_data_01/003/104_003_0019.wav


------------------------------------------------------------
FILE: 104_003_0193.wav
REF: 그 바람에 나의 몸도 겹쳐 쓰러지며 한창 피어난 노란 동백꽃 속으로 폭 파묻혀 버렸다
HYP: 그 바람에 나의 몸도 겹쳐스러지면 한참 피어난 노란 동백꽃속으로 폭파무처버렸다.
PATH: /home/data/bootcamp/user2/data_zeroth/test_data_01/003/104_003_0193.wav


------------------------------------------------------------
FILE: 104_003_0253.wav
REF: 현재 백화점과 영화관 등은 오픈해 영업하고 있고 테마파크 및 아파트 등의 공사는 이천 십 팔 년 완공을 목표로 진행돼 왔다
HYP: 현재 백화점과 영화관 등은 오픈의 영화파고 있고 테마파크미 아파트 등의 공사는 2018년 완공을 목표로 진행되어 왔다.
PATH: /home/data/bootcamp/user2/data_zeroth/test_data_01/003/104_003_0253.wav


------------------------------------------------------------
FILE: 104_003_0280.wav
REF: 또한 박 전 대통령 측은 앞으로 검찰 수사와 관련해 수사 내용에 대한 입장뿐 아니라 변호인단의 활동 상황 등도 소상하게 공개한다는 방침이다
HYP: 또한 박 전 대통령 측은 앞으로 검찰 수사와 관련해 수사 내용에 대한 입장 뿐 아니라 변호인단의 활동 상황 등도 수상하게 공개한다는 방침입니다.
PATH: /home/data/bootcamp/user2/data_zeroth/test_data_01/003/104_003_0280.wav


------------------------------------------------------------
FILE: 104_003_0294.wav
REF: 한 잔의 커피가 소비자에게 전해지기 위해선 여러 차례의 공정을 거쳐야 한다
HYP: 한잔의 커피가 소비자에게 전해지기 위해서는 여러 차례의 공정을 거쳐야 한다.
PATH: /home/data/bootcamp/user2/data_zeroth/test_data_01/003/104_003_0294.wav


------------------------------------------------------------
FILE: 104_003_0352.wav
REF: 다른 지역에서는 그곳이 아무리 넓다 하더라도 푸야 라이몬디를 발견하기가 어려울 것입니다
HYP: 다른 지역에서는 그곳이 아무리 넓다 하더라도 푸야 라인 문제를 발견하기가 어려울 것입니다.
PATH: /home/data/bootcamp/user2/data_zeroth/test_data_01/003/104_003_0352.wav


... (41 outputs left out)

------------------------------------------------------------
FILE: 104_003_1925.wav
REF: 박 대통령은 위민관 삼 층 영상국무회의실에서 참모들과 머리를 맞대고 정국 수습책을 논의한다고 한다
HYP: 박 대통령은 위민관 3층 영상 국무회의실에서 찬모들과 머리를 맞대고 전국 수수책을 논의한다고 한다.
PATH: /home/data/bootcamp/user2/data_zeroth/test_data_01/003/104_003_1925.wav


------------------------------------------------------------
FILE: 104_003_1938.wav
REF: 계열별로는 의약계열 교수를 제외하고 모두 불안하다고 느꼈다
HYP: 개혈별로는 의학계회를 교수를 제외하고 모두 불안하다고 느꼈다.
PATH: /home/data/bootcamp/user2/data_zeroth/test_data_01/003/104_003_1938.wav


------------------------------------------------------------
FILE: 104_003_1968.wav
REF: 견인이 번거로운 고급 수입차와 사 륜구동차는 견인하지 않는다는 불문율이 공공연히 퍼져 있다
HYP: 견인이 번거로운 고급 수입차와 사론부동차는 견인하지 않는다는 불문율이 궁금해니 퍼져있다.
PATH: /home/data/bootcamp/user2/data_zeroth/test_data_01/003/104_003_1968.wav


------------------------------------------------------------
